# CountYOLO v4 — Kaggle Training Notebook

**Yêu cầu trước khi chạy:**
1. Notebook Settings → **Internet: ON**
2. Notebook Settings → **Accelerator: GPU T4 x2** (hoặc P100)
3. Đã upload dataset FSC-147 lên Kaggle Dataset

**Cấu trúc dữ liệu trên Kaggle Dataset:**
```
kaggle_dataset/
└── fsc147/
    ├── images_384_VarV2/
    ├── annotation_FSC147_384.json
    └── Train_Test_Val_FSC_147.json
```
→ Sau khi Add dataset vào notebook, data sẽ nằm tại `/kaggle/input/<tên-dataset>/`

In [ ]:
# Bước 1: Kiểm tra GPU và storage
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {props.name} — {props.total_memory / 1024**3:.1f} GB VRAM')

import shutil
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'\n/kaggle/working — Free: {free // 1024**3} GB / {total // 1024**3} GB')

In [ ]:
# Bước 2: Clone CountYOLO từ GitHub
import os

REPO_DIR = '/kaggle/working/countyolo'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Thien-Dan/countyolo.git $REPO_DIR
else:
    print('Repo da co, pull latest...')
    !git -C $REPO_DIR pull

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# Bước 3: Cài đặt dependencies
!pip install -q transformers scipy wandb tqdm pyyaml pillow opencv-python torchvision
!pip install -q git+https://github.com/facebookresearch/sam2.git
print('\nDependencies installed OK')

In [ ]:
# Bước 4: Tìm và Symlink dataset FSC-147 từ /kaggle/input/
import os, glob

# Tự động tìm annotation_FSC147_384.json trong toàn bộ /kaggle/input/
annotation_files = glob.glob('/kaggle/input/**/annotation_FSC147_384.json', recursive=True)

if annotation_files:
    FSC147_SRC = os.path.dirname(annotation_files[0])
    print(f'Tim thay FSC-147 tai: {FSC147_SRC}')

    FSC147_DST = f'{REPO_DIR}/data/FSC147'
    os.makedirs(f'{REPO_DIR}/data', exist_ok=True)

    if not os.path.exists(FSC147_DST):
        os.symlink(FSC147_SRC, FSC147_DST)
        print(f'Symlinked: {FSC147_SRC} -> {FSC147_DST}')
    else:
        print('Symlink da ton tai')

    # Kiểm tra nhanh
    n_images = len(os.listdir(f'{FSC147_DST}/images_384_VarV2'))
    print(f'So anh FSC-147: {n_images}')
    assert os.path.exists(f'{FSC147_DST}/annotation_FSC147_384.json'), 'Thieu annotation file!'
    assert os.path.exists(f'{FSC147_DST}/Train_Test_Val_FSC_147.json'), 'Thieu split file!'
    print('Dataset OK')
else:
    print('KHONG TIM THAY FSC-147!')
    print('Hay Add Dataset vao notebook:')
    print('  1. Click "+ Add data" (goc phai tren)')
    print('  2. Tim dataset FSC-147 ban da upload')
    print('  3. Restart kernel va chay lai')
    raise FileNotFoundError('FSC-147 dataset not found in /kaggle/input/')

In [ ]:
# Bước 5: Sanity Check — Overfit 1 mini-batch để xác nhận pipeline OK
# Nếu loss giảm dần (không NaN, không Inf) → có thể bắt đầu train thật
!python train.py --sanity-check

In [ ]:
# Bước 6 (Optional): Test LLM Inference path
!python tests/test_llm_inference.py

In [ ]:
# Bước 7: Train thật!
# Checkpoint lưu tại:
#   checkpoint_best.pth   — epoch có epoch_loss thấp nhất
#   checkpoint_latest.pth — epoch cuối cùng
# Sau khi Kaggle session kết thúc, download từ Output tab

import os
print(f'Working dir: {os.getcwd()}')
print('Bat dau training...\n')

!python train.py --config configs/countyolo_fsc147.yaml

In [ ]:
# Bước 8: Kiểm tra và copy checkpoint ra /kaggle/working để download
import shutil, os

# train.py lưu 2 file: checkpoint_best.pth và checkpoint_latest.pth
checkpoints_to_copy = [
    (f'{REPO_DIR}/checkpoint_best.pth',   '/kaggle/working/checkpoint_best.pth'),
    (f'{REPO_DIR}/checkpoint_latest.pth', '/kaggle/working/checkpoint_latest.pth'),
]

copied = 0
for src, dst in checkpoints_to_copy:
    if os.path.exists(src):
        shutil.copy(src, dst)
        size_mb = os.path.getsize(dst) / 1024**2
        print(f'Copied: {os.path.basename(dst)} ({size_mb:.1f} MB)')
        copied += 1
    else:
        print(f'Khong tim thay: {src}')

if copied > 0:
    print(f'\n{copied} checkpoint(s) san sang de download tu Kaggle Output tab!')
else:
    print('Khong co checkpoint nao duoc luu — kiem tra training da chay chua.')